In [0]:
from pyspark.sql import functions as F

catalog = "de_project_"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

In [0]:
# ======= CUSTOMERS =======
print("======= CUSTOMERS PARENT =======")
spark.table(f"{catalog}.{silver_schema}.customers_parent").printSchema()

print("======= CUSTOMERS CHILD =======")
spark.table(f"{catalog}.{silver_schema}.customers_child").printSchema()

======= CUSTOMERS PARENT =======
root
 |-- customer_code: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)

======= CUSTOMERS CHILD =======
root
 |-- customer_code: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)



In [0]:
# ======= PRODUCTS =======
print("======= PRODUCTS PARENT =======")
spark.table(f"{catalog}.{silver_schema}.products_parent").printSchema()

print("======= PRODUCTS CHILD =======")
spark.table(f"{catalog}.{silver_schema}.products_child").printSchema()

======= PRODUCTS PARENT =======
root
 |-- product_code: string (nullable = true)
 |-- division: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- variant: string (nullable = true)

======= PRODUCTS CHILD =======
root
 |-- product_code: string (nullable = true)
 |-- division: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- variant: string (nullable = true)
 |-- product_id: string (nullable = true)



In [0]:
# ======= GROSS PRICE =======
print("======= GROSS PRICE PARENT =======")
spark.table(f"{catalog}.{silver_schema}.gross_price_parent").printSchema()

print("======= GROSS PRICE CHILD =======")
spark.table(f"{catalog}.{silver_schema}.gross_price_child").printSchema()

======= GROSS PRICE PARENT =======
root
 |-- product_code: string (nullable = true)
 |-- price_inr: integer (nullable = true)
 |-- year: integer (nullable = true)

======= GROSS PRICE CHILD =======
root
 |-- product_code: string (nullable = true)
 |-- price_inr: double (nullable = true)
 |-- year: integer (nullable = true)



In [0]:
# ======= ORDERS =======
print("======= ORDERS PARENT =======")
spark.table(f"{catalog}.{silver_schema}.orders_parent").printSchema()

print("======= ORDERS CHILD =======")
spark.table(f"{catalog}.{silver_schema}.orders_child").printSchema()

======= ORDERS PARENT =======
root
 |-- date: date (nullable = true)
 |-- product_code: string (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- sold_quantity: integer (nullable = true)

======= ORDERS CHILD =======
root
 |-- order_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- sold_quantity: double (nullable = true)



**Transformations to be done in Child Orders Data**

In [0]:
from pyspark.sql import functions as F

# Read child orders from silver
orders_child_df = spark.table(f"{catalog}.{silver_schema}.orders_child")

# Aggregate to monthly level before merging with parent
orders_child_agg = orders_child_df\
    .withColumn("date", F.trunc(F.col("date"), "month"))\
    .groupBy("date", "customer_code", "product_code")\
    .agg(F.sum("sold_quantity").alias("sold_quantity"))

display(orders_child_agg)
print("Total rows after aggregation:", orders_child_agg.count())

date,customer_code,product_code,sold_quantity
2025-07-01,789202,3c2039742ba8540af7bb6be1039dd5b2,2944.0
2025-07-01,789321,f7f422b676f90e47165a60e954456553,1742.0
2025-07-01,789303,9850d395a9af6f68ce64cdee3c6c5562,4929.0
2025-07-01,789622,d041f64d8c7033c3f3c927f041882bc6,3810.0
2025-07-01,789402,3b75358f5d88e0eabe11ca1c7e8a0d6c,960.0
2025-07-01,999999,f7f422b676f90e47165a60e954456553,6938.0
2025-07-01,789420,f7f422b676f90e47165a60e954456553,1565.0
2025-07-01,789201,f7f422b676f90e47165a60e954456553,1691.0
2025-07-01,789403,c66a526988f52472967be25f04281690,5872.0
2025-07-01,789421,3174a5138e4d00e72763c018ab51bd27,3318.0


Total rows after aggregation: 3239


In [0]:
# Read parent orders from silver
orders_parent_df = spark.table(f"{catalog}.{silver_schema}.orders_parent")

display(orders_parent_df)
print("Total rows:", orders_parent_df.count())

date,product_code,customer_code,sold_quantity
2024-01-01,ARCHDDE20D,70002017,161
2024-01-01,ARCH158F41,70002017,133
2024-01-01,ARCHAFF0E4,70002017,76
2024-01-01,ARCH6B94F7,70002017,92
2024-01-01,ARCH5D1FE7,70002017,117
2024-01-01,ARCH7B49A9,70002017,36
2024-01-01,ARCH497D34,70002017,98
2024-01-01,ARCHE71D79,70002017,156
2024-01-01,BADM88C384,70002017,28
2024-01-01,BADMA5EBA3,70002017,33


Total rows: 93055


**Remove the unwanted columns from all tables**

In [0]:
# Drop product_id from products child
products_child_df = spark.table(f"{catalog}.{silver_schema}.products_child")\
    .drop("product_id")

products_child_df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{catalog}.{silver_schema}.products_child")

print("product_id dropped successfully!")


product_id dropped successfully!


In [0]:
# Drop order_id from orders child
orders_child_df = spark.table(f"{catalog}.{silver_schema}.orders_child")\
    .drop("order_id")

orders_child_df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{catalog}.{silver_schema}.orders_child")

print("order_id dropped successfully!")

# Validate
spark.table(f"{catalog}.{silver_schema}.orders_child").printSchema()

order_id dropped successfully!
root
 |-- date: date (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- sold_quantity: integer (nullable = true)



**Change the child data datatypes to match the parent data datatypes**

In [0]:
# Fix gross price CHILD - cast price_inr to integer
gross_price_child_df = spark.table(f"{catalog}.{silver_schema}.gross_price_child")\
    .withColumn("price_inr", F.col("price_inr").cast("integer"))

gross_price_child_df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{catalog}.{silver_schema}.gross_price_child")

print("gross_price_child price_inr cast to integer successfully!")

gross_price_child price_inr cast to integer successfully!


In [0]:
# Fix orders CHILD - cast sold_quantity to integer
orders_child_df = spark.table(f"{catalog}.{silver_schema}.orders_child")\
    .withColumn("sold_quantity", F.col("sold_quantity").cast("integer"))

orders_child_df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{catalog}.{silver_schema}.orders_child")

print("orders_child sold_quantity cast to integer successfully!")

orders_child sold_quantity cast to integer successfully!


In [0]:
# ======= CUSTOMERS =======
print("======= CUSTOMERS PARENT =======")
spark.table(f"{catalog}.{silver_schema}.customers_parent").printSchema()

print("======= CUSTOMERS CHILD =======")
spark.table(f"{catalog}.{silver_schema}.customers_child").printSchema()

======= CUSTOMERS PARENT =======
root
 |-- customer_code: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)

======= CUSTOMERS CHILD =======
root
 |-- customer_code: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)



In [0]:
# ======= PRODUCTS =======
print("======= PRODUCTS PARENT =======")
spark.table(f"{catalog}.{silver_schema}.products_parent").printSchema()

print("======= PRODUCTS CHILD =======")
spark.table(f"{catalog}.{silver_schema}.products_child").printSchema()

======= PRODUCTS PARENT =======
root
 |-- product_code: string (nullable = true)
 |-- division: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- variant: string (nullable = true)

======= PRODUCTS CHILD =======
root
 |-- product_code: string (nullable = true)
 |-- division: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- variant: string (nullable = true)



In [0]:
# ======= GROSS PRICE =======
print("======= GROSS PRICE PARENT =======")
spark.table(f"{catalog}.{silver_schema}.gross_price_parent").printSchema()

print("======= GROSS PRICE CHILD =======")
spark.table(f"{catalog}.{silver_schema}.gross_price_child").printSchema()

======= GROSS PRICE PARENT =======
root
 |-- product_code: string (nullable = true)
 |-- price_inr: integer (nullable = true)
 |-- year: integer (nullable = true)

======= GROSS PRICE CHILD =======
root
 |-- product_code: string (nullable = true)
 |-- price_inr: integer (nullable = true)
 |-- year: integer (nullable = true)



In [0]:
print("======= ORDERS PARENT =======")
spark.table(f"{catalog}.{silver_schema}.orders_parent").printSchema()

print("======= ORDERS CHILD =======")
spark.table(f"{catalog}.{silver_schema}.orders_child").printSchema()

======= ORDERS PARENT =======
root
 |-- date: date (nullable = true)
 |-- product_code: string (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- sold_quantity: integer (nullable = true)

======= ORDERS CHILD =======
root
 |-- date: date (nullable = true)
 |-- customer_code: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- sold_quantity: integer (nullable = true)



In [0]:
# Also check row counts
print("Orders Parent count:", spark.table(f"{catalog}.{silver_schema}.orders_parent").count())
print("Orders Child count:", spark.table(f"{catalog}.{silver_schema}.orders_child").count())

Orders Parent count: 93055
Orders Child count: 41579


**Final Check**

In [0]:
# Final validation of all tables
tables = [
    "customers_parent", "customers_child",
    "products_parent", "products_child",
    "gross_price_parent", "gross_price_child",
    "orders_parent", "orders_child"
]

for table in tables:
    print(f"======= {table.upper()} =======")
    spark.table(f"{catalog}.{silver_schema}.{table}").printSchema()

======= CUSTOMERS_PARENT =======
root
 |-- customer_code: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)

======= CUSTOMERS_CHILD =======
root
 |-- customer_code: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- market: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- channel: string (nullable = true)

======= PRODUCTS_PARENT =======
root
 |-- product_code: string (nullable = true)
 |-- division: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- variant: string (nullable = true)

======= PRODUCTS_CHILD =======
root
 |-- product_code: string (nullable = true)
 |-- division: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- variant: string (nullable = true)

======= GROSS_PRICE_PARENT =======
root
